# plotnine 入門 — ggplot2 の文法でグラフを描く

**plotnine** は、R の有名なグラフライブラリ **ggplot2** の文法(グラフィックスの文法、Grammar of Graphics)を Python に移植したライブラリです。
「データ」「対応づけ(aes)」「図形(geom)」を `+` でつなぎ合わせて、グラフを組み立てていきます。

このノートブックはブラウザ上(JupyterLite の Pyodide カーネル)で動くので、環境構築は不要です。

## このノートブックの使い方

- コードセルをクリックして **Shift + Enter** を押すと、そのセルが実行され、次のセルに移動します。
- **上から順番に** 実行してください。前のセルで作ったデータを、後のセルで使うことがあります。
- **最初のセルは plotnine のインストールと日本語フォントの読み込みのため、実行に時間がかかります**(数十秒程度)。`[*]` の表示が数字に変わるまで待ってください。

In [ ]:
import piplite
await piplite.install(["plotnine==0.15.8", "matplotlib-fontja==1.1.0"])

import matplotlib_fontja
import numpy as np
import pandas as pd

matplotlib_fontja.japanize()

plotnine から、このノートブックで使う部品をインポートします。
`from plotnine import *` と書いてすべてを読み込む流儀もありますが、ここでは使うものを明示します。

plotnine はテーマ側のフォント設定(既定は sans-serif)で文字を描くため、日本語を表示するには
`plotnine.options.base_family` に日本語フォント(IPAexGothic)を指定しておきます。

In [ ]:
import plotnine
from plotnine import (
    aes,
    element_text,
    facet_wrap,
    geom_bar,
    geom_boxplot,
    geom_histogram,
    geom_line,
    geom_point,
    geom_smooth,
    ggplot,
    labs,
    theme,
    theme_minimal,
)
from plotnine.data import economics, mpg

plotnine.options.base_family = "IPAexGothic"

print("plotnine", plotnine.__version__)

## サンプルデータ mpg

plotnine には ggplot2 と同じサンプルデータが付属しています。ここでは自動車の燃費データ **mpg**(234 台分)を使います。よく使う列は次の通りです。

| 列名 | 意味 |
|---|---|
| displ | 排気量(リットル) |
| cty | 市街地での燃費(マイル/ガロン) |
| hwy | 高速道路での燃費(マイル/ガロン) |
| class | 車のタイプ(SUV、コンパクトなど) |
| drv | 駆動方式(f: 前輪、r: 後輪、4: 四輪) |

In [ ]:
mpg.head()

## 基本の散布図 — グラフは 3 つの要素でできている

ggplot2 文法では、グラフを次の 3 つの要素の足し算で組み立てます。

1. **データ** … `ggplot(mpg, ...)`
2. **対応づけ(aes)** … どの列を x 軸・y 軸・色などに割り当てるか … `aes(x="displ", y="hwy")`
3. **図形(geom)** … 点・線・棒など、どう描くか … `geom_point()`

排気量と燃費の関係を散布図にしてみましょう。

In [ ]:
(
    ggplot(mpg, aes(x="displ", y="hwy"))
    + geom_point()
)

## 色で情報を追加する

`aes` に `color="class"` を足すと、車のタイプごとに点が色分けされ、凡例も自動で付きます。
`labs()` でタイトルや軸ラベルも付けてみます(日本語も使えます)。

In [ ]:
(
    ggplot(mpg, aes(x="displ", y="hwy", color="class"))
    + geom_point()
    + labs(
        title="排気量と燃費の関係",
        x="排気量(リットル)",
        y="高速道路燃費(マイル/ガロン)",
        color="車のタイプ",
    )
)

## 傾向線を重ねる — geom_smooth

`geom_smooth(method="lm")` を足すと、回帰直線とその信頼区間を重ねられます。
`+` でいくつでも geom を重ねられるのが、この文法の便利なところです。

In [ ]:
(
    ggplot(mpg, aes(x="displ", y="hwy"))
    + geom_point(color="steelblue")
    + geom_smooth(method="lm", color="crimson")
    + labs(title="回帰直線を重ねる", x="排気量(リットル)", y="高速道路燃費")
)

## ファセット — カテゴリ別の小さなグラフに分割する

`facet_wrap("class")` を足すと、車のタイプごとに小さなグラフを並べて描けます。
1 行足すだけでこれができるのは ggplot2 文法ならではです。

In [ ]:
(
    ggplot(mpg, aes(x="displ", y="hwy"))
    + geom_point()
    + facet_wrap("class")
    + labs(title="車のタイプ別に分けて描く", x="排気量(リットル)", y="高速道路燃費")
)

## ヒストグラム

数値の分布を見るには `geom_histogram()` を使います。`bins` で棒の本数を指定します。

In [ ]:
(
    ggplot(mpg, aes(x="hwy"))
    + geom_histogram(bins=20, fill="steelblue", color="white")
    + labs(title="高速道路燃費の分布", x="燃費(マイル/ガロン)", y="台数")
)

## 棒グラフ

`geom_bar()` はカテゴリごとの件数を自動で数えて棒グラフにします。

In [ ]:
(
    ggplot(mpg, aes(x="class", fill="class"))
    + geom_bar(show_legend=False)
    + labs(title="車のタイプ別の台数", x="車のタイプ", y="台数")
)

## 箱ひげ図

カテゴリごとの分布のばらつきを比べるには `geom_boxplot()` が便利です。

In [ ]:
(
    ggplot(mpg, aes(x="class", y="hwy", fill="class"))
    + geom_boxplot(show_legend=False)
    + labs(title="車のタイプ別の燃費のばらつき", x="車のタイプ", y="高速道路燃費")
)

## 折れ線グラフ — 時系列データ

付属データ **economics**(米国の月次経済統計)で、失業者数の推移を折れ線グラフにします。

In [ ]:
economics.head()

In [ ]:
(
    ggplot(economics, aes(x="date", y="unemploy"))
    + geom_line(color="darkorange")
    + labs(title="米国の失業者数の推移", x="年", y="失業者数(千人)")
)

## テーマで見た目を整える

`theme_minimal()` などのテーマを足すと、全体のデザインが変わります。
さらに `theme()` で図のサイズや文字の大きさなど、細かい調整もできます。

In [ ]:
(
    ggplot(mpg, aes(x="displ", y="hwy", color="class"))
    + geom_point()
    + theme_minimal()
    + theme(figure_size=(8, 4), plot_title=element_text(size=14))
    + labs(title="theme_minimal() を適用した散布図", x="排気量(リットル)", y="高速道路燃費", color="車のタイプ")
)

## 実データを読み込んで描く — iris.csv

このサイトの `data/` フォルダーにあるアヤメのデータ `iris.csv` を pandas で読み込みます。
ノートブックから見た相対パスは `../data/iris.csv` です。

このデータには、2 行目の品種名が `se` と欠けて記録されているという小さな「傷」があります(実データにはこうした傷がよくあります)。まず読み込んで、品種ごとの件数を確認してみましょう。

In [ ]:
iris = pd.read_csv("../data/iris.csv")
iris["species"].value_counts()

`se` は `setosa` の書き間違いなので、置き換えて修正してから描きます。

In [ ]:
iris["species"] = iris["species"].replace("se", "setosa")
iris["species"].value_counts()

In [ ]:
(
    ggplot(iris, aes(x="petal_length", y="petal_width", color="species"))
    + geom_point(size=2)
    + labs(
        title="アヤメの花びらの長さと幅",
        x="花びらの長さ (cm)",
        y="花びらの幅 (cm)",
        color="品種",
    )
)

## 練習問題

ここまでの内容を使って、自分でグラフを描いてみましょう。

**練習 1**: mpg データで、市街地燃費 `cty`(x 軸)と高速道路燃費 `hwy`(y 軸)の散布図を描いてください。駆動方式 `drv` で色分けし、日本語のタイトルを付けてみましょう。

In [ ]:
# ここにコードを書いてみましょう


### 解答例 1

In [ ]:
(
    ggplot(mpg, aes(x="cty", y="hwy", color="drv"))
    + geom_point()
    + labs(title="市街地燃費と高速道路燃費", x="市街地燃費", y="高速道路燃費", color="駆動方式")
)

**練習 2**: iris データで、品種 `species` ごとの、がく片の長さ `sepal_length` の箱ひげ図を描いてください。

In [ ]:
# ここにコードを書いてみましょう


### 解答例 2

In [ ]:
(
    ggplot(iris, aes(x="species", y="sepal_length", fill="species"))
    + geom_boxplot(show_legend=False)
    + labs(title="品種ごとのがく片の長さ", x="品種", y="がく片の長さ (cm)")
)

## まとめ

- グラフは「データ + aes + geom」の足し算で組み立てる
- `aes` に `color=` や `fill=` を足すだけで色分けと凡例ができる
- `facet_wrap()` でカテゴリ別の小さなグラフに分割できる
- `labs()` と `theme_*()` で仕上げる(日本語も使える)

もっと知りたい人は公式ドキュメント <https://plotnine.org> を見てみましょう。R の ggplot2 の知識がそのまま生かせます。